# 05 · Churn Proxy and Correlation

**Goal:** estimate how many customers are inactive and compare their behaviour with active customers.

**Note:** the data has no churn label, so *churn* here is a **proxy**: a customer with no order in the 90 days before the last order date in the data. It is an inactivity measure, not a trained churn model.

**Source:** `gold.fact_sales`, `gold.report_customers`

**Output:** churn flags, churn rate by segment, behaviour comparison and a correlation heatmap in `output/`

## 1. Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import config

sns.set_style("whitegrid")
os.makedirs("output", exist_ok=True)

## 2. Load data and set the analysis date

In [53]:
engine = create_engine(config.get_conn_url())
df_sales = pd.read_sql(
    """
    SELECT *
    FROM gold.fact_sales
    WHERE order_date IS NOT NULL
    """,
    engine,
)
df_customers = pd.read_sql("SELECT * FROM gold.report_customers", engine)

df_sales["order_date"] = pd.to_datetime(df_sales["order_date"], errors="coerce")
df_customers["last_orderdate"] = pd.to_datetime(
    df_customers["last_orderdate"], errors="coerce"
)
ANALYSIS_DATE = df_sales["order_date"].max()
print("ANALYSIS_DATE:", ANALYSIS_DATE.date())

ANALYSIS_DATE: 2014-01-28


Analysis date (last order in the data): 2014-01-28.

## 3. Inactivity proxy (> 90 days since last order)

In [73]:
churn_threshold_days = 90
df_customers["days_since_last_order"] = (
    ANALYSIS_DATE - df_customers["last_orderdate"]
).dt.days
df_customers["is_churned_proxy"] = (
    df_customers["days_since_last_order"] > churn_threshold_days
).astype(int)

proxy_rate = (df_customers["is_churned_proxy"].mean()) * 100
print(f"proxy churn rate (> {churn_threshold_days} days inactive): {proxy_rate:.1f}%")

proxy churn rate (> 90 days inactive): 73.3%


73.3% of customers are inactive under this definition. Most customers buy only once, so a high proxy rate is expected. Read it as an inactivity rate, not as true attrition.

## 4. Churn proxy by segment and age group

In [32]:
churn_by_segment = (
    df_customers.groupby("customer_segments")
    .agg(
        customers=("customer_key", "count"),
        proxy_churn_rate_pct=("is_churned_proxy", lambda S: 100 * (S.mean())),
    )
    .reset_index()
)
churn_by_age = (
    df_customers.groupby("age_group")
    .agg(
        customers=("customer_key", "count"),
        proxy_churn_rate_pct=("is_churned_proxy", lambda S: 100 * (S.mean())),
    )
    .reset_index()
)

print(churn_by_segment.round(1))
print(churn_by_age.round(1))

  customer_segments  customers  proxy_churn_rate_pct
0               new      14629                  74.3
1           regular       2200                  67.5
2               vip       1653                  72.2
      age_group  customers  proxy_churn_rate_pct
0         40-49       5636                  74.8
1  50 and above      12846                  72.7


By segment: new 74.3% · vip 72.2% · regular 67.5%. Only two age groups appear (40-49 and 50 and above), because the youngest customer is 40 (age is calculated against today's date).

## 5. Behaviour comparison: active vs. proxy-churned (medians)

In [46]:
behavior_cols = [
    "total_order",
    "total_sales",
    "avg_order_value",
    "total_quantity",
    "total_products",
]
behavior_summary = (
    df_customers.groupby("is_churned_proxy")[behavior_cols]
    .median()
    .T.rename(columns={0: "active_median", 1: "proxy_churned_median"})
)
behavior_summary["ratio_churned_to_active"] = behavior_summary[
    "proxy_churned_median"
] / behavior_summary["active_median"].replace(0, np.nan)
print(round(behavior_summary, 2))

is_churned_proxy  active_median  proxy_churned_median  ratio_churned_to_active
total_order                 1.0                   1.0                     1.00
total_sales               183.5                 549.0                     2.99
avg_order_value            97.0                 549.0                     5.66
total_quantity              3.0                   3.0                     1.00
total_products              3.0                   3.0                     1.00


Proxy-churned customers have a higher median total sales (549 vs. 183.5) and AOV (549 vs. 97). Median orders, quantity and product count are the same for both groups.

## 6. Correlation between behavioural metrics

Spearman correlation is used because the metrics are skewed. Derived labels such as the churn flag are left out of the correlation on purpose.

In [48]:
corr_cols = [
    "total_order",
    "total_sales",
    "avg_order_value",
    "total_quantity",
    "total_products",
]
corr = df_customers[corr_cols].corr(method="spearman")
print(corr.round(2))

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", center=0)
plt.title("Spearman Correlation — Customer Behavioral Metrics")
plt.tight_layout()
plt.savefig("output/08_behavior_correlation_heatmap_v2.png", dpi=120)
plt.close()

                 total_order  total_sales  avg_order_value  total_quantity  \
total_order             1.00         0.61             0.39            0.65   
total_sales             0.61         1.00             0.96            0.49   
avg_order_value         0.39         0.96             1.00            0.36   
total_quantity          0.65         0.49             0.36            1.00   
total_products          0.64         0.49             0.36            1.00   

                 total_products  
total_order                0.64  
total_sales                0.49  
avg_order_value            0.36  
total_quantity             1.00  
total_products             1.00  


![Spearman correlation heatmap](output/08_behavior_correlation_heatmap_v2.png)

`total_sales` and `avg_order_value` are strongly related (0.96) because AOV is derived from sales. `total_quantity` and `total_products` move almost together (1.00).

## 7. Save outputs

In [74]:
df_customers[
    [
        "customer_key",
        "customer_number",
        "customer_segments",
        "days_since_last_order",
        "is_churned_proxy",
    ]
].to_csv("output/churn_proxy_flags_v2.csv", index=False)
churn_by_segment.to_csv("output/churn_rate_by_segment_v2.csv", index=False)
behavior_summary.to_csv("output/churn_behavior_comparison_v2.csv")
print("Churn-proxy analysis complete.")

Churn-proxy analysis complete.


## 8. Scratch checks

The SQL `recency` column counts months from the day the view is queried (`GETDATE()`), so it is not used for the 90-day proxy. The proxy uses `last_orderdate` and the last order date in the data.

In [68]:
(df_customers["recency"] < 90).sum()

np.int64(0)

In [70]:
print(df_sales.columns)
print(df_customers.columns)

Index(['order_number', 'product_key', 'customer_key', 'order_date',
       'shipping_date', 'due_date', 'sales_amount', 'quantity', 'price'],
      dtype='str')
Index(['customer_key', 'customer_number', 'customer_name', 'age', 'age_group',
       'customer_segments', 'recency', 'total_order', 'total_sales',
       'avg_order_value', 'avg_monthly_sales', 'total_quantity',
       'total_products', 'last_orderdate', 'lifespan'],
      dtype='str')
